In [34]:
# Chap 15.riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time


In [35]:

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
#print("=" *100)
#print(" 이 크롤러는 서울시민 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
#print("=" *100)
#query_txt = input('1.수집할 자료의 키워드는 무엇입니까?: ')


In [36]:

#Step 3. 수집된 데이터를 저장할 파일 이름 입력 받기
f_dir = input("2.파일을 저장할 폴더명만 쓰세요(기본값:c:\\py_temp\\):")
if f_dir == '' :
    f_dir="c:\\py_temp\\"


In [37]:

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://eungdapso.seoul.go.kr/main.do'
driver.get(url)
driver.maximize_window()
time.sleep(2)


In [38]:

#Step 5. 자동으로 검색어 입력 후 조회하기
#element = driver.find_element(By.ID,'query')
#element.send_keys(query_txt)
#element.send_keys("\n")


#link = soup.find('div','my_current_count_box').find('a')['href']

#//*[@id="content"]/div[2]/a/div

In [39]:

#Step 6.학위 논문 선택하기
#driver.find_element(By.LINK_TEXT,'학위논문').click()
#time.sleep(2)



driver.find_element(By.XPATH,'//*[@id="content"]/div[2]/a/div').click()
time.sleep(2)

In [40]:
# 시장에게 바란다 접속 완료 

In [41]:

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')


In [42]:

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력 받기
import math
total_cnt = soup_1.find('div','rp_lkment').find('span','rplk_count').get_text()
print('키워드 %s (으)로 총 %s 건의 민원신청이 검색되었습니다' %(query_txt,total_cnt))
cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
page_cnt = math.ceil(cnt / 10)
print('%s 건의 데이터를 수집하기 위해 %s 페이지의 게시물을 조회합니다.' %(cnt,page_cnt))
print("\n")


키워드  (으)로 총 340건 건의 민원신청이 검색되었습니다
15 건의 데이터를 수집하기 위해 2 페이지의 게시물을 조회합니다.




여기까지는 잘 되는듯? 

In [43]:

#Step 9. 데이터 수집하기
no2=[] # 게시글 번호 컬럼
title2=[ ] # 게시글 제목 컬럼
date2=[ ] # 게시글 날짜 컬럼
contents2=[] # 상담내용
full_url2=[] # 논문 원본 URL

no = 1 # 게시글 번호 초기값

for a in range(1,page_cnt+1) :
    print("\n")
    print("%s 페이지 내용 수집 시작합니다 =======================" %a)

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    content_list = soup.find('div','rp_tb_wrap default_skin complaint_mayor_hope_list').find_all('div','rp_inlink_item')

    for i in content_list:
        # 여기서 바로 들어 가야겠네?? 아니면 제목 뽑고 들어가거나 ?? 

        # 민원신청 제목 체크하기
        try:
            title=i.find('div','rp_inlink_item').get_text().strip()
        except :
            continue
        else :
            # 1.게시글 번호
            print("\n")
            print("%s 번째 정보를 추출하고 있습니다============" %no)
            no2.append(no)
            print("1.번호 : %s" %no)

            # 2. 민원신청 제목
            title2.append(title.strip())
            print("2.제목 : %s" %title.strip())

            # 3. 민원목록 안의 상세 내역에서 추출할 수 있음.
            url_1 = i.find('div','rp_inlink_item').find('a')['href']
            full_url = 'http://www.riss.kr'+url_1
            time.sleep(1)
            driver.get(full_url)

            html_1 = driver.page_source
            soup_1 = BeautifulSoup(html_1, 'html.parser')

            # 4. 공개일 
            try :
                date_1 =i.find('div','sclinkage_else').find_all('span')
                date_2 = date_1[1].get_text().strip()
            except :
                date_2='공개일가 없습니다'
                date2.append(date_2)
                print("4.공개일 : %s" %date_2)
            else :
                date2.append(date_2)
                print("4.공개일 : %s" %date_2)

            # 5. 상담내용 
#            try :
#                cont=soup_1.find("span","rp_indata").find('div','restd_incont').get_text().replace("\n","").strip()
#            except :
#                cont='상담내용이 없습니다'
#                contents2.append(cont)
#                print("5.상담내용 : %s" %cont)
#            else :
#                contents2.append(cont)
#                print("5.상담내용 : %s" %cont)

            try :
                cont=soup_1.find("div","res_hcell restd").find('div','restd_incont').get_text().replace("\n","").strip()
            except :
                cont='상담내용이 없습니다'
                contents2.append(cont)
                print("5.상담내용 : %s" %cont)
            else :
                contents2.append(cont)
                print("5.상담내용 : %s" %cont)



            driver.back() # 이전 페이지로 돌아가기

            time.sleep(2)

            no += 1

            if no > cnt :
                break

    a += 1
    b = str(a)

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
    except :
        driver.find_element(By.LINK_TEXT,'다음 페이지로').click()

print("요청하신 작업이 모두 완료되었습니다")




1 페이지 내용 수집 시작합니다 =======================


2 페이지 내용 수집 시작합니다 =======================
요청하신 작업이 모두 완료되었습니다


In [44]:

# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
# 현재 날짜와 시간으로 폴더 만들고 파일 이름 설정하기
import os

n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+'RISS'+'-'+s+'-'+'학위논문')

fc_name = f_dir+'RISS'+'-'+s+'-'+'학위논문'+'\\'+'RISS'+'-'+s+'-'+'학위논문'+'.csv'
fx_name = f_dir+'RISS'+'-'+s+'-'+'학위논문'+'\\'+'RISS'+'-'+s+'-'+'학위논문'+'.xlsx'

# 데이터 프레임 생성 후 xlsx, csv 형식으로 저장하기
import pandas as pd

no2=[] # 게시글 번호 컬럼
title2=[ ] # 게시글 제목 컬럼
date2=[ ] # 게시글 날짜 컬럼
contents2=[] # 상담내용
full_url2=[] # 논문 원본 URL

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['날짜']=pd.Series(date2)
df['초록(논문일경우)']=pd.Series(contents2)
df['자료URL주소']=pd.Series(full_url2)

# xls 형태로 저장하기
df.to_excel(fx_name,index=False)

# csv 형태로 저장하기
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

요청하신 데이터 수집 작업이 정상적으로 완료되었습니다


참고

In [45]:
#spans = i.find('p','etc').find_all('span')

#author = spans[0].text
#company = spans[1].text
#date = spans[2].text
#degree = spans[3].text

In [46]:
#1️⃣ 태그로 찾기
#find('div')
#2️⃣ class로 찾기
#find('span','writer')
#3️⃣ 여러개 찾고 인덱스로 구분
#find_all('span')[2]